# 03 — Bayesian MCMC Regime Model

Upgrades the classical model to full Bayesian estimation via MCMC (PyMC / NUTS sampler).

Key outputs:
- **Posterior distributions** over regime parameters (not just point estimates)
- **Uncertainty-aware regime probabilities** — confidence score for position sizing
- **Credible intervals** on regime means and volatilities

**Prerequisites:** Run `01_eda.ipynb` and `02_regime_baseline.ipynb` first.

⚠️ Full MCMC sampling can take 15–30 minutes. Use `sample_vi()` for quick exploration.

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.data.loader import get_price_data, load_config
from src.models.bayesian_regime import BayesianRegimeSwitching
from src.utils.plotting import RegimePlotter

%matplotlib inline
config = load_config('../config/settings.yaml')
config['paths']['figures'] = '../results/figures/'
config['paths']['processed_data'] = '../data/processed/'
print("Ready")

In [ ]:
prices = get_price_data(config)
returns = np.log(prices['index'] / prices['index'].shift(1)).dropna()
print(f"Observations: {len(returns)}")

## Step 1: Quick Exploration with Variational Inference

ADVI converges in minutes. Use this to validate model structure before committing to full MCMC.

In [ ]:
model = BayesianRegimeSwitching(n_regimes=3, config=config)

# VI for quick exploration — set use_full_mcmc=True below when ready
USE_FULL_MCMC = False

if USE_FULL_MCMC:
    model.sample(returns)
else:
    model.sample_vi(returns, n_iterations=30000)
    print("\nVI complete. Set USE_FULL_MCMC=True for full posterior inference.")

## Step 2: Posterior Regime Probabilities

In [ ]:
regime_probs = model.posterior_regime_probs()
confidence = model.position_confidence()
uncertainty = model.regime_uncertainty()

print("Regime probabilities (last 5 rows):")
print(regime_probs.tail(5).round(3))
print()
print(f"Average confidence: {confidence.mean():.2%}")
print(f"Days below confidence threshold ({config['strategy']['min_regime_confidence']:.0%}): "
      f"{(confidence < config['strategy']['min_regime_confidence']).sum()} "
      f"({(confidence < config['strategy']['min_regime_confidence']).mean():.1%})")

In [ ]:
# Overview plot
plotter = RegimePlotter(config)
_ = plotter.regime_overview(prices['index'], regime_probs, confidence, save=True)

## Step 3: Posterior Parameter Distributions

In [ ]:
# Posterior plots (requires full MCMC trace for full visualisation)
model.plot_posterior(config, save=True)

## Step 4: Compare Classical vs Bayesian

In [ ]:
from src.data.loader import load_config
import pandas as pd
from pathlib import Path

processed = Path('../data/processed/')

try:
    classical_probs = pd.read_csv(processed / 'regime_probs_classical.csv',
                                   index_col='date', parse_dates=True)

    common = regime_probs.index.intersection(classical_probs.index)
    bayesian_regime = regime_probs.reindex(common).idxmax(axis=1)
    classical_regime = classical_probs.reindex(common).idxmax(axis=1)

    agreement = (bayesian_regime == classical_regime).mean()
    print(f"Regime agreement between Classical and Bayesian: {agreement:.1%}")
    print()
    print("Bayesian regime counts:")
    print(bayesian_regime.value_counts())
    print()
    print("Classical regime counts:")
    print(classical_regime.value_counts())

except FileNotFoundError:
    print("Run notebook 02 first to generate classical results.")

In [ ]:
# Save Bayesian results for backtest
model.save_results(config)
print("Bayesian regime results saved.")